## 🎯 Learning Objectives
* Understand the practical steps involved in setting up a pretraining pipeline for a small language model.
* Implement a basic transformer-decoder-only architecture suitable for next-token prediction.
* Develop a data loading and tokenization pipeline for a domain-specific corpus.
* Execute a fundamental pretraining loop and monitor its progress.


## Exercise: Pretrain a Small Language Model on a Domain Corpus

**Lesson ID:** FT02-L08

**Objective:** In this exercise, you will implement a simplified pretraining pipeline for a small language model (LLM) on a synthetic domain-specific corpus. The goal is to solidify your understanding of the core components involved in LLM pretraining, from data preparation to model architecture and the training loop itself.

While real-world LLM pretraining involves massive datasets, complex distributed systems, and highly optimized architectures, this exercise focuses on the fundamental building blocks. You will build a miniature version to grasp the mechanics.

### Task Description

Your task is to implement a complete pretraining pipeline for a small, decoder-only transformer model. This includes:

1.  **Data Preparation:** Loading a mock domain corpus, tokenizing it, and preparing it into batches suitable for training.
2.  **Model Architecture:** Defining a simple transformer-decoder-only model (e.g., 2-4 layers, small embedding dimension, few attention heads).
3.  **Training Loop:** Implementing a basic pretraining loop using next-token prediction as the objective.
4.  **Evaluation:** Tracking the training loss over time.

### Requirements

*   **Data:** Utilize the provided mock domain corpus. You should tokenize it using a basic tokenizer (e.g., a character-level tokenizer or a simple BPE tokenizer from `transformers`).
*   **Model:** Your model should be a decoder-only transformer. It must include:
    *   Token embeddings.
    *   Positional embeddings.
    *   Multiple transformer decoder blocks (each with multi-head self-attention and a feed-forward network).
    *   A final linear layer for next-token prediction.
*   **Training:** Implement a standard training loop. Use a suitable optimizer (e.g., AdamW) and a learning rate scheduler (e.g., constant or linear warmup).
*   **Code Quality:** Your code should be well-structured, readable, and include comments where necessary.

### Evaluation Criteria

*   **Correctness:** The data pipeline correctly processes the input, and the model architecture is a functional decoder-only transformer.
*   **Functionality:** The training loop runs without errors and shows a decreasing loss over epochs.
*   **Clarity:** Code is easy to understand and follows best practices.
*   **Efficiency (Basic):** While not the primary focus, avoid obvious inefficiencies in your implementation.

Good luck!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import random
import math

# --- Configuration and Hyperparameters (2026-ready defaults) ---
# Using a small configuration to ensure quick execution for an exercise
# In 2026, even 'small' models might be larger, but for pedagogical clarity,
# we keep dimensions manageable.

# Device configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Model Hyperparameters
vocab_size = 5000  # Will be determined by tokenizer, but set a max for embedding layer
block_size = 128   # Maximum context length for predictions
n_layer = 4        # Number of transformer blocks
n_head = 4         # Number of attention heads
n_embd = 256       # Embedding dimension
dropout = 0.1      # Dropout rate

# Training Hyperparameters
batch_size = 32
learning_rate = 3e-4
max_iters = 2000   # Number of training iterations
eval_interval = 200
eval_iters = 100

# --- Mock Domain Corpus Generation ---
# For this exercise, we'll generate a synthetic corpus that mimics a specific domain.
# Let's imagine a 'Quantum Computing Research Paper' domain.

def generate_quantum_corpus(num_documents=100, doc_length=500):
    keywords = [
        "quantum entanglement", "superposition", "qubit", "quantum gate",
        "quantum supremacy", "decoherence", "quantum algorithm", "quantum computing",
        "quantum mechanics", "Bell state", "Grover's algorithm", "Shor's algorithm",
        "quantum error correction", "quantum annealing", "quantum cryptography",
        "photon", "spin", "wave function", "Hamiltonian", "Schrödinger equation"
    ]
    templates = [
        "Recent advancements in {} have shown promising results.",
        "The phenomenon of {} is central to modern {}.",


In [ ]:
        "Understanding {} is crucial for developing robust {}.",


In [ ]:
        "Our research explores the implications of {} in {} systems.",
        "The theoretical framework of {} provides insights into {}.",


In [ ]:
        "Future applications of {} include {} and {}.",


In [ ]:
        "Experimental verification of {} confirms the principles of {}.",


In [ ]:
        "The challenge of {} must be addressed for scalable {}.",


In [ ]:
        "This paper investigates the role of {} in achieving {}.",


In [ ]:
        "We propose a novel approach to mitigate {} in {} architectures."
    ]

    corpus = []
    for _ in range(num_documents):
        doc = []
        for _ in range(doc_length // 10): # Generate shorter sentences to build up doc_length
            num_keywords = random.randint(1, 3)
            selected_keywords = random.sample(keywords, num_keywords)
            template = random.choice(templates)
            try:
                sentence = template.format(*selected_keywords)
            except IndexError: # Handle cases where template needs more keywords than selected
                sentence = template.format(*selected_keywords, *random.sample(keywords, 3 - num_keywords))
            doc.append(sentence)
        corpus.append(" ".join(doc) + ".")
    return "\n".join(corpus)

print("Generating mock quantum computing corpus...")
raw_text = generate_quantum_corpus(num_documents=500, doc_length=1000)
print(f"Corpus generated. Total characters: {len(raw_text)}")

# --- Tokenization ---
# In 2026, custom tokenizers for domain-specific tasks are common.
# For simplicity, we'll use a pre-trained BPE tokenizer and train it on our corpus.
# This simulates a domain-adapted tokenizer.

# Using a ByteLevelBPETokenizer from the `tokenizers` library directly for better control
# This is more realistic for domain-specific pretraining than AutoTokenizer directly.
from tokenizers import ByteLevelBPETokenizer

# Initialize a tokenizer
tokenizer = ByteLevelBPETokenizer()

# Train the tokenizer on our corpus
print("Training tokenizer...")
# Create a temporary file for the tokenizer training
with open("temp_corpus.txt", "w", encoding="utf-8") as f:
    f.write(raw_text)

tokenizer.train(
    files=["temp_corpus.txt"],
    vocab_size=vocab_size, # Use the defined vocab_size
    min_frequency=2,
    special_tokens=["<unk>", "<s>", "</s>", "<pad>"]
)

# Save the tokenizer (optional, but good practice)
tokenizer.save_model(".", "quantum_tokenizer")

# Load it back using transformers AutoTokenizer for easy integration
# This step bridges `tokenizers` library with `transformers` ecosystem.
from transformers import PreTrainedTokenizerFast

# Load the trained tokenizer
# Ensure the special tokens are correctly mapped
fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="quantum_tokenizer-vocab.json",
    merges_file="quantum_tokenizer-merges.txt",
    unk_token="<unk>",
    bos_token="<s>",
    eos_token="</s>",
    pad_token="<pad>"
)

vocab_size = fast_tokenizer.vocab_size # Update vocab_size based on trained tokenizer
print(f"Tokenizer trained. New vocab size: {vocab_size}")

# Encode the entire corpus
encoded_corpus = fast_tokenizer.encode(raw_text).ids
data = torch.tensor(encoded_corpus, dtype=torch.long)
print(f"Corpus encoded. Total tokens: {len(data)}")

# --- Data Loader Helper Functions ---

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data_split = data # For simplicity, use the full data for both train/val in this exercise
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print("Setup complete. Ready for model implementation.")


## Student Implementation: Build and Train Your MiniGPT

Now it's your turn! Implement the `MiniGPT` model and the training loop based on the requirements outlined above. You should define the following:

1.  **`Head`**: A single self-attention head.
2.  **`MultiHeadAttention`**: Combines multiple `Head`s.
3.  **`FeedForward`**: A simple two-layer MLP.
4.  **`Block`**: A transformer block combining `MultiHeadAttention` and `FeedForward` with layer normalization and residual connections.
5.  **`MiniGPT`**: The main model class, integrating token and positional embeddings, a stack of `Block`s, and a final language modeling head.
6.  **Training Loop**: The main loop to train your `MiniGPT` model using the `get_batch` and `estimate_loss` helper functions provided.

Remember to use `torch.nn.Module` for all your components and ensure they are moved to the correct `device`.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# --- Reference Solution: MiniGPT Model Implementation ---

class Head(nn.Module):
    """ One head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # Compute attention scores ("affinities")
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5 # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # Decoder-only mask
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)

        # Perform the weighted aggregation of the values
        v = self.value(x) # (B, T, head_size)
        out = wei @ v # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd) # Projection back to residual stream
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ A simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # Expansion factor of 4 is common
            nn.GELU(), # Modern activation function
            nn.Linear(4 * n_embd, n_embd), # Projection back
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Residual connections and layer normalization are crucial for deep networks
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    """ A small, decoder-only Transformer Language Model """

    def __init__(self):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Initialize weights for better training stability
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)
        x = self.blocks(x) # (B, T, C)
        x = self.ln_f(x) # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Reshape for F.cross_entropy: (N, C) and (N)
            logits = logits.view(-1, logits.shape[-1])
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets, ignore_index=fast_tokenizer.pad_token_id)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # Get the predictions
            logits, loss = self(idx_cond)
            # Focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

# --- Instantiate Model and Optimizer ---

model = MiniGPT()
model = model.to(device)

# In 2026, `torch.compile` is a standard optimization for PyTorch models.
# It compiles the model into optimized kernels, significantly speeding up training.
# For this small model, the benefits might be less dramatic but it's good practice.
# model = torch.compile(model) # Uncomment for potential speedup if PyTorch 2.0+ is available

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"Model instantiated with {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters.")

# --- Training Loop ---

print("Starting training...")
for iter in range(max_iters):

    # Every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Training complete!")

# --- Optional: Generate some text after training ---
print("\nGenerating text sample after training:")
# Start with a beginning-of-sequence token
context = torch.tensor(fast_tokenizer.encode("<s>The quantum state of").ids, dtype=torch.long, device=device).unsqueeze(0)
generated_tokens = model.generate(context, max_new_tokens=100)[0].tolist()
print(fast_tokenizer.decode(generated_tokens))

# Clean up temporary tokenizer files
import os
if os.path.exists("temp_corpus.txt"):
    os.remove("temp_corpus.txt")
if os.path.exists("quantum_tokenizer-vocab.json"):
    os.remove("quantum_tokenizer-vocab.json")
if os.path.exists("quantum_tokenizer-merges.txt"):
    os.remove("quantum_tokenizer-merges.txt")
print("Temporary tokenizer files cleaned up.")
